# Análise exploratória sobre o banco do pipeline

**Projeto Integrador (P15) · Fundamentos de Banco de Dados (BDED-2026.1)**
Especialização em Projetos em Inteligência Artificial · UNIFAP Digital
Aluno: Sandro · Prof. Dr. Adolfo Colares

Repositório: <https://github.com/xsansil/pipeline-churn-ecommerce>

---

## O que este caderno faz

Explora o banco **depois** que o pipeline rodou, percorrendo as quatro camadas na
ordem em que o dado as atravessa:

```
bronze   o dado como chegou das quatro fontes
silver   limpo, tipado e enriquecido
gold     uma linha por cliente, pronta para o modelo
meta     o registro de tudo o que aconteceu
```

Quase toda informação aqui sai de **SQL sobre o banco**, e não de arquivo
intermediário. É uma escolha deliberada: se o pipeline cumpriu o papel dele,
tudo o que importa tem de estar consultável no PostgreSQL.

## Antes de executar

O banco precisa estar populado. Da raiz do repositório:

```bash
python pipeline.py
```

São cerca de 14 segundos, do CSV bruto ao modelo treinado. Depois disso, este
caderno roda do começo ao fim sem depender de mais nada.

## 1. Conexão

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import pandas as pd

# o caderno vive em notebooks/, e o código do projeto está um nível acima
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

from src import db

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 40)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

VERDE, RUST, ESCURO, CINZA = "#2E6B52", "#B0402C", "#16222E", "#8A95A1"

eng = db.engine()
consulta = lambda sql, **p: db.le(sql, eng, **p)
print("Conectado.")

A conexão usa `src/db.py`, o mesmo módulo que o ETL e a API utilizam. A senha
vem do `.env` ou do ambiente, nunca do código, e a URL aparece em log com a
senha mascarada.

## 2. Panorama: as quatro camadas

In [ ]:
consulta("""
    select table_schema as camada, table_name as tabela,
           (xpath('/row/c/text()', query_to_xml(
              format('select count(*) c from %I.%I', table_schema, table_name),
              false, true, '')))[1]::text::int as linhas
      from information_schema.tables
     where table_schema in ('bronze','silver','gold','meta')
       and table_type = 'BASE TABLE'
     order by array_position(array['bronze','silver','gold','meta'], table_schema),
              table_name
""")

A consulta conta as linhas de cada tabela sem escrever uma consulta por tabela.
O `query_to_xml` executa o `count(*)` montado em tempo de execução e devolve o
resultado como XML, de onde o `xpath` extrai o número. É um truque conhecido do
PostgreSQL para inventariar um esquema inteiro de uma vez.

Cada camada tem um papel:

| Camada | Papel | Restrições |
|---|---|---|
| `bronze` | o dado como chegou da fonte | nenhuma, tudo `TEXT` |
| `silver` | limpo, tipado, enriquecido | chave primária, `CHECK`, índices |
| `gold` | pronto para consumo, um cliente por linha | chave primária |
| `meta` | proveniência e histórico | somente acréscimo |

## 3. Bronze: o dado como chegou

In [ ]:
consulta("""
    select 'transacoes (CSV)'        as fonte, count(*) as registros from bronze.transacoes
    union all select 'catalogo (JSON)',         count(*) from bronze.catalogo
    union all select 'municipios (API IBGE)',   count(*) from bronze.municipios_ibge
    union all select 'clientes (PostgreSQL)',   count(*) from bronze.clientes
""")

Quatro fontes de tipos diferentes. A do IBGE é a única externa de verdade, vinda
de uma API pública, e por isso a coleta dela tem cache em disco e degrada por UF
quando a rede falha.

In [ ]:
consulta("""
    select 'data inválida ou ausente' as problema,
           count(*) filter (where data !~ '^[0-9]{4}-[0-9]{2}-[0-9]{2}$') as linhas
      from bronze.transacoes
    union all select 'valor com vírgula decimal', count(*) filter (where valor like '%,%')    from bronze.transacoes
    union all select 'valor vazio',               count(*) filter (where valor = '')          from bronze.transacoes
    union all select 'idade vazia',               count(*) filter (where idade = '')          from bronze.transacoes
    union all select 'cidade em caixa alta',      count(*) filter (where cidade <> '' and cidade = upper(cidade)) from bronze.transacoes
    union all select 'duplicata exata',           count(*) - count(distinct transacao_id)     from bronze.transacoes
    order by 2 desc
""")

A bronze guarda tudo como `TEXT`, sem chave e sem restrição, **de propósito**.
É o que permite ingerir `"31/02/2024"` e `"450,00"` sem o banco recusar.

Duas razões para essa camada existir:

**Reprocessar sem voltar à fonte.** Se a regra de limpeza mudar amanhã, basta
refazer a etapa seguinte. A fonte pode estar fora do ar ou já ter mudado.

**Preservar a evidência.** Depois da limpeza ninguém conseguiria provar que havia
185 duplicatas no arquivo original. A consulta acima só é possível porque o dado
sujo continua guardado.

## 4. Silver: o que a limpeza e a integração fizeram

O funil da limpeza não precisa ser recalculado: ele ficou **registrado no banco**
enquanto o pipeline rodava. A tabela `meta.etl_execucoes` guarda cada passo de
cada etapa, e nada nela é apagado.

In [ ]:
consulta("""
    with ultima as (
      select max(momento) as fim from meta.etl_execucoes where etapa = 'silver'
    )
    select fonte as etapa, quantidade as removidos, detalhe
      from meta.etl_execucoes, ultima
     where etapa = 'silver' and fase = 'LIMPEZA'
       and momento > ultima.fim - interval '10 minutes'
     order by id
""")

De 6.383 linhas restaram 5.809 transações, 9,0% descartado.

A etapa que exigiu mais critério foi a última. O método estatístico padrão para
outlier, o intervalo interquartil sobre o valor, removeria **587** registros,
liderados por monitor, SSD e headset, que são os produtos caros e legítimos do
catálogo. Ele não distingue "caro" de "errado".

O critério adotado compara o **preço unitário** com o teto do catálogo: o item
mais caro custa R$ 900, então qualquer preço unitário acima de R$ 1.500 não
corresponde a produto nenhum. Isolou 72 registros, exatamente os erros de
digitação.

In [ ]:
consulta("""
    with ultima as (
      select max(momento) as fim from meta.etl_execucoes where etapa = 'silver'
    )
    select fonte as juncao, quantidade as casaram, detalhe
      from meta.etl_execucoes, ultima
     where etapa = 'silver' and fase = 'JUNCAO'
       and momento > ultima.fim - interval '10 minutes'
     order by id
""")

Cada junção tem contador, e isso não é zelo excessivo.

`LEFT JOIN` **nunca levanta erro**. Quando a chave não casa, ele preenche nulo e
segue. Um join que deveria casar tudo e casa pouco devolve uma tabela do tamanho
certo, com as colunas certas, cheia de nulos, e o problema só aparece lá na
frente como "o modelo não aprende". O contador transforma isso em uma linha no
log.

In [ ]:
consulta("""
    select 'transações'              as indicador,
           (select count(*) from bronze.transacoes)::text as bronze,
           (select count(*) from silver.transacoes)::text as silver
    union all
    select 'e-mails inválidos',
           (select count(*) from bronze.transacoes
             where email !~ '^[^@[:space:]]+@[^@[:space:]]+\\.[^@[:space:]]+$')::text,
           (select count(*) from silver.transacoes where not email_valido)::text
    union all
    select 'sem cidade',
           (select count(*) from bronze.transacoes where cidade = '' or cidade is null)::text,
           (select count(*) from silver.transacoes where cidade = 'Desconhecida')::text
    union all
    select 'com mesorregião', '0',
           (select count(*) from silver.transacoes where mesorregiao is not null)::text
    union all
    select 'com margem', 'não calculável',
           (select count(*) from silver.transacoes where margem is not null)::text
""")

As três últimas linhas são o argumento da arquitetura multi-fonte. Mesorregião e
margem **não existem** no extrato de vendas: a primeira veio da API do IBGE, a
segunda do catálogo em JSON, que traz o custo.

E as duas primeiras mostram o efeito do cadastro do CRM: o que a limpeza de
fonte única teria de adivinhar, a integração forneceu.

## 5. O negócio, visto pela camada silver

In [ ]:
cat = consulta("""
    select categoria,
           count(*)                                 as transacoes,
           round(sum(valor))                        as receita,
           round(sum(margem))                       as margem,
           round(100 * sum(margem) / sum(valor), 1) as margem_pct,
           round(avg(valor), 2)                     as ticket_medio
      from silver.transacoes
     group by categoria
     order by receita desc
""")
cat

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))

ordem = cat.iloc[::-1]
ax[0].barh(ordem["categoria"], ordem["receita"], color=VERDE)
ax[0].set_title("Receita por categoria")
ax[0].xaxis.set_major_formatter(mtick.StrMethodFormatter("R$ {x:,.0f}"))

ax[1].barh(ordem["categoria"], ordem["margem_pct"], color=RUST)
ax[1].set_title("Margem por categoria")
ax[1].xaxis.set_major_formatter(mtick.PercentFormatter())

plt.tight_layout()
plt.show()

A inversão entre os dois gráficos é o achado desta seção. **Monitores lideram a
receita e têm a pior margem; acessórios são o contrário.** Quem olhasse só o
faturamento concluiria que monitor é o produto mais importante.

Essa leitura só é possível porque o custo veio do catálogo. Sem a segunda fonte,
o gráfico da direita não existiria.

In [ ]:
serie = consulta("""
    select to_char(date_trunc('month', data), 'YYYY-MM') as mes,
           count(*)                   as transacoes,
           count(distinct cliente_id) as clientes,
           round(sum(valor))          as receita
      from silver.transacoes
     group by 1 order by 1
""")

fig, ax1 = plt.subplots(figsize=(13, 4))
ax1.bar(serie["mes"], serie["receita"], color=VERDE, alpha=.75, label="receita")
ax1.set_ylabel("receita")
ax1.yaxis.set_major_formatter(mtick.StrMethodFormatter("R$ {x:,.0f}"))
ax1.tick_params(axis="x", rotation=90)

ax2 = ax1.twinx()
ax2.plot(serie["mes"], serie["clientes"], color=RUST, marker="o", lw=2,
         label="clientes ativos")
ax2.set_ylabel("clientes que compraram")
ax2.grid(False)

ax1.set_title("Receita e clientes ativos por mês")
fig.legend(loc="upper right", bbox_to_anchor=(.9, .88))
plt.tight_layout()
plt.show()

A linha de clientes ativos cai ao longo do período enquanto a receita oscila sem
tendência tão clara. **Essa queda é o churn aparecendo de forma agregada**, e é
o fenômeno que o modelo vai tentar antecipar no nível de cada cliente.

In [ ]:
reg = consulta("""
    select coalesce(mesorregiao, 'Sem região') as mesorregiao,
           count(distinct cliente_id)          as clientes,
           count(*)                            as transacoes,
           round(sum(valor))                   as receita
      from silver.transacoes
     group by 1 order by receita desc
""")

fig, ax = plt.subplots(figsize=(11, 4))
r = reg.iloc[::-1]
ax.barh(r["mesorregiao"], r["receita"], color=ESCURO)
ax.set_title("Receita por mesorregião (enriquecimento vindo da API do IBGE)")
ax.xaxis.set_major_formatter(mtick.StrMethodFormatter("R$ {x:,.0f}"))
plt.tight_layout()
plt.show()

reg

A mesorregião não vem no extrato de vendas. Ela foi obtida casando o nome do
município com a resposta da API do IBGE, usando como chave o texto **sem acento
e em caixa baixa**: o IBGE devolve "Macapá" e o extrato traz "MACAPA", "  macapa"
ou "Macapa  ". Sem normalizar, o join casaria quase nada e não reclamaria.

In [ ]:
valores = consulta("select valor from silver.transacoes")["valor"].astype(float)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(valores, bins=40, color=VERDE, edgecolor="white")
ax[0].set_title("Distribuição do valor das transações")
ax[0].set_xlabel("valor")

ax[1].boxplot(valores, vert=False, widths=.6)
ax[1].set_title("O mesmo dado em boxplot")
ax[1].set_xlabel("valor")
ax[1].set_yticks([])

plt.tight_layout()
plt.show()

print("mediana R$ %.2f · média R$ %.2f · máximo R$ %.2f"
      % (valores.median(), valores.mean(), valores.max()))

Distribuição assimétrica à direita, com concentração nos itens baratos e uma
cauda de produtos caros. É o formato esperado em varejo, e é exatamente por isso
que o intervalo interquartil não servia como critério de outlier: ele trataria a
cauda legítima como erro.

In [ ]:
cruz = consulta("""
    select canal, forma_pagamento, count(*) as n
      from silver.transacoes
     group by 1, 2 order by 1, 2
""")
tab = cruz.pivot(index="canal", columns="forma_pagamento", values="n").fillna(0)

fig, ax = plt.subplots(figsize=(9, 3.6))
im = ax.imshow(tab.values, cmap="Greens", aspect="auto")
ax.set_xticks(range(len(tab.columns)), tab.columns, rotation=20, ha="right")
ax.set_yticks(range(len(tab.index)), tab.index)
for i in range(tab.shape[0]):
    for j in range(tab.shape[1]):
        ax.text(j, i, int(tab.values[i, j]), ha="center", va="center", fontsize=9)
ax.set_title("Transações por canal e forma de pagamento")
ax.grid(False)
plt.tight_layout()
plt.show()

A distribuição é próxima da uniforme, sem concentração em nenhuma combinação.
Isso é um indício de que canal e forma de pagamento **não carregam padrão de
comportamento**, e a seção 7 vai confirmar isso de forma quantitativa.

## 6. Gold: de transações para clientes

Aqui a granularidade muda. O modelo não classifica compras, classifica pessoas,
então 5.809 transações viram 399 clientes, cada um com 44 colunas que resumem o
histórico dele.

A decisão mais delicada do projeto está nesta camada: **o que conta como churn**.

A definição intuitiva ("mais de 90 dias sem comprar") tem um defeito fatal se a
variável "dias sem comprar" também virar feature, porque a resposta fica dentro
da pergunta. O modelo acerta quase tudo sem ter aprendido nada.

A solução foi cortar o tempo em duas janelas:

```
observação   2023-01-01 .. 2024-09-30   constrói as features
avaliação    2024-10-01 .. 2024-12-31   define o alvo
```

A pergunta usa dados de antes do corte; a resposta vem do que aconteceu depois.

In [ ]:
alvo = consulta("""
    select case churn when 1 then 'Churn' else 'Ativo' end as situacao,
           count(*) as clientes,
           round(100.0 * count(*) / sum(count(*)) over (), 1) as pct
      from gold.features_clientes
     group by churn order by churn
""")

fig, ax = plt.subplots(figsize=(5, 3.4))
ax.bar(alvo["situacao"], alvo["clientes"], color=[VERDE, RUST])
for i, r in alvo.iterrows():
    ax.text(i, r["clientes"], "%d\n%.1f%%" % (r["clientes"], r["pct"]),
            ha="center", va="bottom")
ax.set_title("Distribuição do alvo")
ax.set_ylim(0, alvo["clientes"].max() * 1.25)
plt.tight_layout()
plt.show()

Desbalanceamento leve, com a classe positiva em maioria. Guarde esse número: ele
é a razão pela qual a escolha da métrica, na seção 8, não é trivial.

In [ ]:
consulta("""
    select origem,
           count(*)                                           as features,
           round(max(abs(corr_com_churn))::numeric, 3)        as maior_correlacao,
           round(avg(abs(corr_com_churn))::numeric, 3)        as correlacao_media,
           count(*) filter (where abs(corr_com_churn) > 0.30) as relevantes
      from gold.features_dicionario
     group by origem order by maior_correlacao desc
""")

Esta consulta responde, com SQL, a pergunta que justifica a arquitetura
multi-fonte. E a resposta **não é a esperada**.

As features vindas do catálogo, da API do IBGE e do CRM quase não se movem com o
alvo. A única forte, `margem_total`, correlaciona 0,969 com `total_gasto`: é o
mesmo dado reescalado, não informação nova.

A conclusão honesta é que **a integração melhorou muito a qualidade do dado e
quase nada o poder preditivo**. São coisas diferentes, e misturá-las seria
vender o projeto por algo que ele não fez.

In [ ]:
corr = consulta("""
    select feature, origem, corr_com_churn
      from gold.features_dicionario
     order by abs(corr_com_churn) desc limit 12
""")

fig, ax = plt.subplots(figsize=(10, 5))
c = corr.iloc[::-1]
ax.barh(c["feature"], c["corr_com_churn"],
        color=[RUST if v > 0 else VERDE for v in c["corr_com_churn"]])
ax.axvline(0, color=ESCURO, lw=1)
ax.set_title("As 12 features mais correlacionadas com o churn")
ax.set_xlabel("correlação de Pearson")
plt.tight_layout()
plt.show()

corr

O padrão é coerente: em verde, tudo que indica atividade recente e segura o
cliente; em vermelho, o que indica afastamento.

Nenhuma passa de 0,51 em valor absoluto, e isso é saudável. Uma correlação de
0,9 com o alvo, num problema de churn, quase nunca é um achado: é a resposta
escondida na pergunta. Um teste automatizado do projeto barra qualquer feature
que ultrapasse 0,90.

Vale destacar a `razao_recencia_intervalo`, construída especificamente para este
problema. A recência isolada não distingue dois casos opostos: quem compra a
cada 15 dias e está há 90 sem comprar está seis ciclos fora do padrão **dele**;
quem compra a cada 120 dias e está há 90 está dentro do normal. A recência é a
mesma nos dois, 90 dias. A razão distingue.

## 7. O modelo, lido do banco

As métricas de todos os modelos de todas as rodadas ficam em `meta.treinos`,
agrupadas por um identificador de execução. Nada aqui é recalculado: o treino
gravou, este caderno lê.

In [ ]:
mod = consulta("""
    select modelo, rodadas, auc_medio, cv_auc_medio, f1_medio,
           acuracia_media, vezes_escolhido
      from gold.vw_desempenho_modelos
""")
mod

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
m = mod.iloc[::-1]
y = range(len(m))
ax.barh([i + .2 for i in y], m["auc_medio"].astype(float), height=.4,
        color=VERDE, label="ROC AUC no teste")
ax.barh([i - .2 for i in y], m["cv_auc_medio"].astype(float), height=.4,
        color=CINZA, label="ROC AUC na validação cruzada")
ax.axvline(0.5, color=RUST, ls="--", lw=1.5, label="acaso (0,500)")
ax.set_yticks(list(y), m["modelo"])
ax.set_xlim(0.4, 1.0)
ax.set_title("Desempenho por algoritmo")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()

### Por que a escolha usou ROC AUC, e não F1

A linha de base deste problema, que responde "churn" para todo mundo e não
aprendeu nada, obtém **F1 de 0,718**. Ela acerta todos os que de fato deram
churn, então o recall é 1,0, e como a classe positiva é maioria a precisão já
nasce alta. A média harmônica disso supera a maioria dos modelos treinados.

O F1 enxerga apenas a classe positiva e nunca olha para os verdadeiros
negativos. Selecionar por ele levaria à escolha errada.

Em ROC AUC a mesma linha de base marca exatamente **0,500**, porque atribui a
mesma probabilidade a todos e não ordena ninguém. É ela a métrica de seleção.

In [ ]:
dec = consulta("""
    select decil, clientes, no_teste, pct_churn_geral, pct_churn_teste, risco_medio
      from gold.vw_ganho_por_decil
""")

fig, ax = plt.subplots(figsize=(11, 4))
x = dec["decil"].astype(int)
ax.bar(x - .2, dec["pct_churn_geral"].astype(float), width=.4,
       color=CINZA, label="todos os clientes")
ax.bar(x + .2, dec["pct_churn_teste"].astype(float), width=.4,
       color=RUST, label="somente o conjunto de teste")
ax.set_xticks(x)
ax.set_xlabel("decil de risco (1 = maior risco previsto)")
ax.set_ylabel("% que de fato deu churn")
ax.set_title("O modelo ordena a carteira?")
ax.legend()
plt.tight_layout()
plt.show()

dec

Este é o gráfico que diz para que serve o modelo.

Ele não adivinha quem vai embora. O que ele faz é **ordenar a carteira**: no
decil de maior risco praticamente todos saíram, no de menor risco quase ninguém.
Uma equipe de retenção que não consegue abordar 399 pessoas aborda as 40 do
topo, e acerta a maioria.

A coluna **somente o conjunto de teste é a que vale**. A geral inclui os 299
clientes que o modelo viu durante o treino e sai otimista. No teste o gradiente
continua existindo, mas é visivelmente mais ruidoso, porque são de 5 a 15
clientes por decil.

In [ ]:
consulta("""
    select posicao, nome, cidade, probabilidade, faixa_de_risco,
           total_gasto, churn_real
      from gold.vw_clientes_risco
     order by posicao limit 10
""")

A entrega operacional do projeto é esta lista. A view
`gold.vw_clientes_risco` usa funções de janela (`ROW_NUMBER` para a posição na
fila e `NTILE(10)` para o decil) e já junta o nome do cliente, a região e o
valor histórico, que é o que a operação precisa para decidir a quem ligar
primeiro.

## 8. Conclusões

**O pipeline fecha.** Quatro fontes de tipos diferentes entram, atravessam quatro
camadas no PostgreSQL e saem como uma lista ordenada de clientes em risco. Todo
o percurso é reproduzível com um comando e em cerca de 14 segundos.

**A limpeza é onde se ganha ou se perde.** De 6.383 linhas restaram 5.809, e a
decisão que mais importou não foi estatística: trocar o intervalo interquartil
pelo preço unitário contra o teto do catálogo evitou descartar 515 vendas
legítimas.

**A integração entregou qualidade, não previsão.** Idades imputadas caíram de
296 para 4 e cidades desconhecidas de 170 para 1, mas nenhuma feature vinda das
fontes auxiliares move o alvo de forma relevante. Está registrado assim na
entrega porque é o que o dado mostra.

**A métrica depende do formato do problema.** O F1 de 0,718 da linha de base é a
lembrança mais útil deste trabalho: um número que parece bom isoladamente pode
não significar nada sem a referência ao lado.

**O valor do modelo está na ordenação.** Com ROC AUC de 0,73 ele não classifica
bem, mas concentra os casos reais no topo da lista, que é o que permite agir com
uma equipe pequena.

### Limitações

São 399 clientes, dos quais 100 no teste, então diferenças de poucos pontos
entre modelos cabem na margem de incerteza. As transações são sintéticas,
geradas por script com semente fixa; a coleta do IBGE é real. O corte temporal
é fixo em 2024-10-01, e três meses de janela de avaliação equilibram dois riscos
opostos: uma janela curta classificaria como churn quem apenas demorou a voltar,
e uma longa empobreceria as features.